# Build a MATH-Only Full-Trajectory Solver Dataset

This notebook creates exactly one normalized output file:

```text
normalized_outputs/solver_full_trajectory_dataset.jsonl
```

It loads `EleutherAI/hendrycks_math`, extracts 2,000 unique MATH problems with full solution trajectories and final answers, and writes them in the schema used by `newest_solver`.

Each output row has this shape:

```json
{
  "problem": "...",
  "solution_steps": ["step 1", "step 2", "step 3"],
  "final_answer": "...",
  "example_index": 137,
  "source": "math",
  "level": "Level 3",
  "category": "Algebra"
}
```


## Install Dependencies


In [51]:
# Install older datasets to support dataset python scripts removed in datasets v3.0+.
!pip -q install "datasets<3.0" pandas tqdm


## Imports And Configuration


In [52]:
from __future__ import annotations

import json
import re
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm


# Compatibility patch: datasets<3.0 can hit a dill/pickle legacy-cache bug on Python 3.14+.
# The legacy cache probe is only for reusing old cache locations, so disabling it is safe here.
def patch_datasets_legacy_cache_for_python314() -> None:
    import sys

    if sys.version_info < (3, 14):
        return
    try:
        from datasets.builder import DatasetBuilder
    except Exception:
        return

    def _skip_legacy_cache2(self, dataset_module):
        return None

    DatasetBuilder._check_legacy_cache2 = _skip_legacy_cache2


patch_datasets_legacy_cache_for_python314()

OUTPUT_DIR = Path("normalized_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSONL = OUTPUT_DIR / "solver_full_trajectory_dataset.jsonl"
TARGET_MATH_PROBLEMS = 2000

MATH_CONFIGS = [
    "algebra",
    "counting_and_probability",
    "geometry",
    "intermediate_algebra",
    "number_theory",
    "prealgebra",
    "precalculus",
]
MATH_SPLITS = ["train", "test"]

print(f"Output file: {OUTPUT_JSONL.resolve()}")
print(f"Target MATH examples: {TARGET_MATH_PROBLEMS:,}")


Output file: /Users/timli/MathSolverLLMs/normalized_outputs/solver_full_trajectory_dataset.jsonl
Target MATH examples: 2,000


## Normalization Helpers


In [53]:
def first_present(d: Dict[str, Any], keys: List[str], default=None):
    for key in keys:
        if key in d and d[key] not in (None, ""):
            return d[key]
    return default


def clean_text(x: Any) -> str:
    if x is None:
        return ""
    text = str(x)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    lines = [line.strip() for line in text.split("\n")]
    return "\n".join(lines).strip()


def split_solution_into_steps(text: str) -> List[str]:
    text = clean_text(text)
    if not text:
        return []

    normalized = text.replace("\\\\\n", "\n")
    lines = [line.strip() for line in normalized.split("\n") if line.strip()]
    numbered_like = []
    for line in lines:
        if re.match(r"^(\(?\d+[\).\:]|[-*\u2022]|Step\s+\d+[:.\-])", line, flags=re.I):
            numbered_like.append(re.sub(r"\s+", " ", line).strip())
    if len(numbered_like) >= 2:
        return numbered_like

    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    if len(paragraphs) >= 2:
        return paragraphs

    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\\(])", text)
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]
    if len(sentences) >= 2:
        return sentences

    return [text]


def extract_last_braced_content(text: str, marker: str) -> str:
    start = text.rfind(marker)
    if start == -1:
        return ""
    i = start + len(marker)
    depth = 1
    chars = []
    while i < len(text) and depth > 0:
        char = text[i]
        if char == "{":
            depth += 1
            chars.append(char)
        elif char == "}":
            depth -= 1
            if depth > 0:
                chars.append(char)
        else:
            chars.append(char)
        i += 1
    return "".join(chars).strip() if depth == 0 else ""


def extract_final_answer(text: str) -> str:
    text = clean_text(text)
    if not text:
        return ""

    boxed = extract_last_braced_content(text, r"\boxed{")
    if boxed:
        return boxed

    fboxed = extract_last_braced_content(text, r"\fbox{")
    if fboxed:
        return fboxed

    patterns = [
        r"(?i)final answer\s*(?:is|:)\s*(.+)$",
        r"(?i)therefore[, ]+the answer\s*(?:is|:)\s*(.+)$",
        r"(?i)thus[, ]+the answer\s*(?:is|:)\s*(.+)$",
        r"(?i)answer\s*[:]\s*(.+)$",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.M)
        if match:
            return match.group(1).strip().rstrip(".")

    return ""


def normalize_math_record(rec: Dict[str, Any], example_index: int) -> Optional[Dict[str, Any]]:
    problem = clean_text(first_present(rec, ["problem", "question"]))
    solution = clean_text(first_present(rec, ["solution", "answer"]))
    final_answer = clean_text(first_present(rec, ["final_answer", "answer_only"], default=""))
    final_answer = final_answer or extract_final_answer(solution)
    solution_steps = split_solution_into_steps(solution)

    if not problem or not solution_steps or not final_answer:
        return None

    return {
        "problem": problem,
        "solution_steps": solution_steps,
        "final_answer": final_answer,
        "example_index": example_index,
        "source": "math",
        "level": first_present(rec, ["level"], default=""),
        "category": first_present(rec, ["type", "subject", "category"], default=""),
    }


## Load And Normalize MATH


In [54]:
all_records = []
seen_problems = set()
skipped_without_final_answer = 0

for config in MATH_CONFIGS:
    if len(all_records) >= TARGET_MATH_PROBLEMS:
        break

    math_ds = load_dataset("EleutherAI/hendrycks_math", config, trust_remote_code=True)

    for split in MATH_SPLITS:
        if len(all_records) >= TARGET_MATH_PROBLEMS:
            break
        if split not in math_ds:
            continue

        for rec in tqdm(math_ds[split], desc=f"Normalizing MATH/{config}/{split}"):
            if len(all_records) >= TARGET_MATH_PROBLEMS:
                break

            item = normalize_math_record(rec, len(all_records))
            if item is None:
                skipped_without_final_answer += 1
                continue

            problem_key = item["problem"]
            if problem_key in seen_problems:
                continue

            seen_problems.add(problem_key)
            item["example_index"] = len(all_records)
            all_records.append(item)

if len(all_records) < TARGET_MATH_PROBLEMS:
    raise ValueError(
        f"Needed {TARGET_MATH_PROBLEMS} MATH examples with final answers, "
        f"but only found {len(all_records)}. Skipped {skipped_without_final_answer} rows without usable final answers."
    )

print(f"Loaded {len(all_records):,} MATH examples")
print(f"Skipped rows without usable final answers: {skipped_without_final_answer:,}")
print("Category counts:")
display(pd.Series([record["category"] for record in all_records]).value_counts())
print("Level counts:")
display(pd.Series([record["level"] for record in all_records]).value_counts())


Normalizing MATH/algebra/train:   0%|          | 0/1744 [00:00<?, ?it/s]

Normalizing MATH/algebra/test:   0%|          | 0/1187 [00:00<?, ?it/s]

Loaded 2,000 MATH examples
Skipped rows without usable final answers: 2
Category counts:


Algebra    2000
Name: count, dtype: int64

Level counts:


Level 5    497
Level 4    460
Level 3    449
Level 2    385
Level 1    209
Name: count, dtype: int64

## Export The Single Output File


In [55]:
with OUTPUT_JSONL.open("w", encoding="utf-8") as f:
    for record in all_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

assert len(all_records) == TARGET_MATH_PROBLEMS
assert Counter(record["source"] for record in all_records) == {"math": TARGET_MATH_PROBLEMS}

print(f"Wrote {len(all_records):,} MATH examples to {OUTPUT_JSONL}")


Wrote 2,000 MATH examples to normalized_outputs/solver_full_trajectory_dataset.jsonl


## Inspect A Few Normalized Examples


In [56]:
for record in all_records[:5]:
    print("=" * 100)
    print("source:", record["source"])
    print("category:", record.get("category", ""))
    print("level:", record.get("level", ""))
    print("problem:", record["problem"][:400])
    print("num_steps:", len(record["solution_steps"]))
    print("first_step:", record["solution_steps"][0][:300] if record["solution_steps"] else "")
    print("final_answer:", record.get("final_answer", ""))


source: math
category: Algebra
level: Level 5
problem: Let \[f(x) = \left\{
\begin{array}{cl} ax+3, &\text{ if }x>2, \\
x-5 &\text{ if } -2 \le x \le 2, \\
2x-b &\text{ if } x <-2.
\end{array}
\right.\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).
num_steps: 6
first_step: For the piecewise function to be continuous, the cases must "meet" at $2$ and $-2$.
final_answer: 0
source: math
category: Algebra
level: Level 5
problem: A rectangular band formation is a formation with $m$ band members in each of $r$ rows, where $m$ and $r$ are integers. A particular band has less than 100 band members. The director arranges them in a rectangular formation and finds that he has two members left over. If he increases the number of members in each row by 1 and reduces the number of rows by 2, there are exactly enough places in the n
num_steps: 8
first_step: Let $x$ be the number of band members in each row for t